# Notebook 03 — Supplementary Tables 3–9: Full Replication

Runs all nine MATLAB supplementary tables in one notebook, using only precomputed `.npz` files.
All statistical functions are defined inline.

| Table | Dataset | Analysis |
|-------|---------|----------|
| 3 | Shekhar | Difficulty dependence (contrast 3 vs 1) |
| 4 | Rouault Expt 1 | Difficulty dependence (high vs low contrast) |
| 5 | Rouault Expt 2 | Difficulty dependence |
| 6 | Haddara | Metacognitive bias (Xue recoding) |
| 7 | Maniscalco | Metacognitive bias |
| 8 | Shekhar | Metacognitive bias |
| 9 | Locke | Response bias RM-ANOVA |

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')

# ── locate repository root and add src/ to path ──────────────
REPO = os.path.abspath(os.path.join(
    os.getcwd(), '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
DATA = os.path.join(REPO, 'matlab', 'metasignal_mat', 'Preprocess', 'orig_csv_files')
OUT  = os.path.join(REPO, 'notebooks', 'precomputed')
sys.path.insert(0, os.path.join(REPO, 'src'))
os.makedirs(OUT, exist_ok=True)

import numpy as np
import pandas as pd
from scipy import stats
from metasignal.stdpy.compute_all import compute_all_measures
print("metasignal loaded successfully.")


metasignal loaded successfully.


In [2]:
MEASURE_NAMES = [
    "meta-d'", "AUC2", "Gamma", "Phi", "DeltaConf",
    "M-Ratio", "AUC2-Ratio", "Gamma-Ratio", "Phi-Ratio", "DeltaConf-Ratio",
    "M-Diff",  "AUC2-Diff",  "Gamma-Diff",  "Phi-Diff",  "DeltaConf-Diff",
    "meta-noise", "meta-uncertainty",
    "d'", "Criterion", "Confidence",
]
N_MEAS = 20


## Statistical helper functions

In [3]:
# ── t-test matching MATLAB perform_ttest.m ───────────────────
def ttest_1samp(data):
    """One-sample t-test vs 0. Returns (t, df, p, cohen_d, ci_lo, ci_hi).
    Cohen's d = t/sqrt(n) matching MATLAB: Cohen_d = t/sqrt(df+1)."""
    x = np.asarray(data, float)
    x = x[~np.isnan(x)]
    n = len(x)
    if n < 2:
        return (np.nan,)*6
    t, p = stats.ttest_1samp(x, 0)
    d    = t / np.sqrt(n)
    sem  = x.std(ddof=1) / np.sqrt(n)
    ci   = x.mean() + stats.t.ppf([0.025, 0.975], n-1) * sem
    return t, n-1, p, d, ci[0], ci[1]

def p_stars(p):
    if np.isnan(p): return ''
    if p < 0.001:   return '***'
    if p < 0.01:    return '**'
    if p < 0.05:    return '*'
    return 'ns'

# ── one-way repeated-measures ANOVA ─────────────────────────
def rm_anova_1way(data_2d):
    """data_2d: (n_subjects, n_conditions). Returns (F, df_b, df_e, p, eta2_p)."""
    n, k   = data_2d.shape
    grand  = np.nanmean(data_2d)
    row_m  = np.nanmean(data_2d, axis=1, keepdims=True)
    col_m  = np.nanmean(data_2d, axis=0, keepdims=True)
    ss_b   = n * np.sum((col_m - grand)**2)
    ss_s   = k * np.sum((row_m - grand)**2)
    ss_e   = np.sum((data_2d - grand)**2) - ss_b - ss_s
    df_b, df_e = k-1, (n-1)*(k-1)
    F  = (ss_b/df_b) / (ss_e/df_e)
    p  = stats.f.sf(F, df_b, df_e)
    return F, df_b, df_e, p, ss_b/(ss_b+ss_e)

# ── 3-SD outlier removal per level per measure ───────────────
def remove_3sd_outliers(arr):
    """arr: (n_sub, n_levels, n_meas). Matches MATLAB ana_taskPerformance.m."""
    out = arr.copy()
    _, n_lev, n_meas = out.shape
    for m in range(n_meas):
        for lv in range(n_lev):
            col = out[:, lv, m]
            mu, sd = np.nanmean(col), np.nanstd(col, ddof=1)
            if not np.isnan(mu) and sd > 0:
                out[(col < mu-3*sd) | (col > mu+3*sd), lv, m] = np.nan
        bad = np.isnan(out[:, :, m]).any(axis=1)
        out[bad, :, m] = np.nan
    return out

print("Statistical helper functions defined.")


Statistical helper functions defined.


## Load precomputed data

In [4]:
sh_diff = np.load(os.path.join(OUT, 'shekhar_mle.npz'))['diff']   # (20,3,20)
r1_diff = np.load(os.path.join(OUT, 'rouault1_mle.npz'))['diff']  # (466,2,20)
r2_diff = np.load(os.path.join(OUT, 'rouault2_mle.npz'))['diff']  # (484,2,20)
ha_npz  = np.load(os.path.join(OUT, 'haddara_mle.npz'))
ma_npz  = np.load(os.path.join(OUT, 'maniscalco_mle.npz'))
lo_rb   = np.load(os.path.join(OUT, 'locke_mle.npz'))['rb']       # (10,7,20)
ha_bias = ha_npz['bias']   # (70,2,20)
ma_bias = ma_npz['bias']   # (22,2,20)
sh_bias_full = np.load(os.path.join(OUT, 'shekhar_mle.npz'))['bias']  # (20,3,2,20)
print("Data loaded.")


Data loaded.


## Supp Tables 3–5 — Difficulty dependence

**Method**: Per subject compute measures at hard vs easy difficulty, apply 3SD outlier removal per measure per level, then one-sample t-test on easy − hard difference.

In [5]:
cohens_d_lbl = "Cohen's d"
def difficulty_table(diff_arr, label, reported={}):
    """diff_arr: (n_sub, 2, 20) [hard=0, easy=1]. Applies 3SD, runs t-tests."""
    clean = remove_3sd_outliers(diff_arr)
    delta = clean[:, 1, :] - clean[:, 0, :]   # easy - hard
    print(f"\nSupplementary Table — {label}")
    print("=" * 72)
    print(f"  {'Measure':<20} {'t':>7} {'df':>5} {'p':>8} {'sig':>4} {cohens_d_lbl:>9}")
    print("  " + "-"*62)
    for m, name in enumerate(MEASURE_NAMES):
        t, df, p, d, lo, hi = ttest_1samp(delta[:, m])
        rep_t = reported.get(name, np.nan)
        match = "" if np.isnan(t) else ("✓" if abs(t-rep_t)<abs(rep_t)*0.05+0.1 else "~") if not np.isnan(rep_t) else ""
        t_s  = f"{t:7.3f}" if not np.isnan(t) else "    NaN"
        d_s  = f"{d:9.3f}" if not np.isnan(d) else "      NaN"
        df_s = f"{df:5.0f}" if not np.isnan(df) else "  NaN"
        p_s  = f"{p:8.4f}" if not np.isnan(p) else "     nan"
        print(f"  {name:<20} {t_s} {df_s} {p_s} {p_stars(p):>4} {d_s}  {match}")

# Table 3: Shekhar (contrast 1=hard, contrast 3=easy → indices 0 and 2)
difficulty_table(sh_diff[:, [0,2], :], "Table 3: Shekhar (n=20) — contrast 3 vs 1",
    reported={"meta-d'":22.616,"AUC2":20.612,"Gamma":29.238,"Phi":10.898,
              "DeltaConf":14.834,"M-Ratio":-1.240,"AUC2-Ratio":-3.215,
              "M-Diff":-4.087,"AUC2-Diff":-4.006,"d'":23.777,"Criterion":0.166,"Confidence":14.543})

# Table 4: Rouault1 (low=0, high=1)
difficulty_table(r1_diff, "Table 4: Rouault 2018 Expt 1 (n=466) — high vs low contrast",
    reported={"meta-d'":35.285,"AUC2":35.405,"d'":49.278,"Confidence":32.390})

# Table 5: Rouault2
difficulty_table(r2_diff, "Table 5: Rouault 2018 Expt 2 (n=484) — high vs low contrast",
    reported={"meta-d'":15.304,"AUC2":13.657,"d'":48.583,"Confidence":15.334})



Supplementary Table — Table 3: Shekhar (n=20) — contrast 3 vs 1
  Measure                    t    df        p  sig Cohen's d
  --------------------------------------------------------------
  meta-d'               22.732    19   0.0000  ***     5.083  ✓
  AUC2                  20.728    19   0.0000  ***     4.635  ✓
  Gamma                 29.393    19   0.0000  ***     6.572  ✓
  Phi                   10.926    19   0.0000  ***     2.443  ✓
  DeltaConf             14.909    19   0.0000  ***     3.334  ✓
  M-Ratio               -1.168    19   0.2573   ns    -0.261  ✓
  AUC2-Ratio            -3.015    19   0.0071   **    -0.674  ✓
  Gamma-Ratio           -0.408    19   0.6876   ns    -0.091  
  Phi-Ratio             -0.862    19   0.3996   ns    -0.193  
  DeltaConf-Ratio       -1.290    19   0.2125   ns    -0.288  
  M-Diff                -3.980    19   0.0008  ***    -0.890  ✓
  AUC2-Diff             -3.838    19   0.0011   **    -0.858  ✓
  Gamma-Diff            -3.002    19   0.007

  Phi-Diff              -5.053   480   0.0000  ***    -0.230  
  DeltaConf-Diff        -6.726   471   0.0000  ***    -0.310  
  meta-noise               NaN   NaN      nan            NaN  
  meta-uncertainty         NaN   NaN      nan            NaN  
  d'                    48.583   469   0.0000  ***     2.241  ✓
  Criterion              2.407   482   0.0165    *     0.110  
  Confidence            15.334   476   0.0000  ***     0.702  ✓


## Supp Tables 6–8 — Metacognitive bias (Xue recoding)

Xue et al. (2021) recoding: apply two transformations to confidence ratings, compute measures under each, test recode2 − recode1 against zero.

In [6]:
cohens_d_lbl = "Cohen's d"
def xue_recode(conf, rtype):
    """rtype 1 = high-confidence bias (subtract 1, floor at min+1).
       rtype 2 = low-confidence bias  (replace max with max-1)."""
    valid = conf[~np.isnan(conf)]
    if len(np.unique(valid)) < 3: return np.full_like(conf, np.nan)
    c = conf.copy().astype(float)
    if rtype == 1:
        c -= 1; cmin = np.nanmin(c); c[c == cmin] = cmin + 1
    else:
        cmax = np.nanmax(c); c[c == cmax] = cmax - 1
    return c

def bias_table(bias_arr, label, reported={}):
    """bias_arr: (n_sub, 2, 20) [recode1=0, recode2=1]. t-test on recode2-recode1."""
    delta = bias_arr[:, 1, :] - bias_arr[:, 0, :]
    EXCL  = {"d'", "Criterion"}
    print(f"\nSupplementary Table — {label}")
    print("=" * 72)
    print(f"  {'Measure':<20} {'t':>7} {'df':>5} {'p':>8} {'sig':>4} {cohens_d_lbl:>9}")
    print("  " + "-"*62)
    for m, name in enumerate(MEASURE_NAMES):
        if name in EXCL: continue
        t, df, p, d, lo, hi = ttest_1samp(delta[:, m])
        rep_t = reported.get(name, np.nan)
        match = "" if np.isnan(t) else ("✓" if abs(t-rep_t)<abs(rep_t)*0.05+0.1 else "~") if not np.isnan(rep_t) else ""
        t_s  = f"{t:7.3f}" if not np.isnan(t) else "    NaN"
        d_s  = f"{d:9.3f}" if not np.isnan(d) else "      NaN"
        df_s = f"{df:5.0f}" if not np.isnan(df) else "  NaN"
        p_s  = f"{p:8.4f}" if not np.isnan(p) else "     nan"
        print(f"  {name:<20} {t_s} {df_s} {p_s} {p_stars(p):>4} {d_s}  {match}")

bias_table(ha_bias, "Table 6: Haddara (n=70)",
    reported={"meta-d'":2.584,"AUC2":0.688,"Gamma":-4.331,"Phi":1.257,
              "DeltaConf":1.034,"M-Ratio":1.994,"Gamma-Diff":2.334,"Confidence":24.538})
bias_table(ma_bias, "Table 7: Maniscalco (n=22)",
    reported={"meta-d'":2.711,"AUC2":3.794,"Phi":5.262,"DeltaConf":5.242,"Confidence":17.328})
sh_bias = np.nanmean(sh_bias_full, axis=1)   # average over 3 contrasts → (20,2,20)
bias_table(sh_bias, "Table 8: Shekhar (n=20)",
    reported={"AUC2":2.804,"Gamma":-4.284,"Phi":5.133,"DeltaConf-Ratio":2.992,"Confidence":13.845})



Supplementary Table — Table 6: Haddara (n=70)
  Measure                    t    df        p  sig Cohen's d
  --------------------------------------------------------------
  meta-d'                2.318    69   0.0234    *     0.277  ~
  AUC2                   0.688    69   0.4936   ns     0.082  ✓
  Gamma                 -4.331    69   0.0000  ***    -0.518  ✓
  Phi                    1.257    69   0.2129   ns     0.150  ✓
  DeltaConf              1.034    69   0.3047   ns     0.124  ✓
  M-Ratio                1.795    69   0.0770   ns     0.215  ✓
  AUC2-Ratio             1.176    69   0.2436   ns     0.141  
  Gamma-Ratio            0.510    69   0.6117   ns     0.061  
  Phi-Ratio              1.062    69   0.2920   ns     0.127  
  DeltaConf-Ratio        1.664    69   0.1006   ns     0.199  
  M-Diff                 2.316    69   0.0236    *     0.277  
  AUC2-Diff              1.237    69   0.2203   ns     0.148  
  Gamma-Diff             2.361    69   0.0210    *     0.282  ✓
 

## Supp Table 9 — Response bias (Locke)

One-way repeated-measures ANOVA across 7 response-bias conditions (F(6,54)).

In [7]:
print("\nSupplementary Table 9 — Response bias: Locke (n=10, 7 conditions)")
print("=" * 72)
print(f"  {'Measure':<20} {'F(6,54)':>9} {'p':>8} {'sig':>4} {'η²p':>7}")
print("  " + "-"*55)
REPORTED_T9 = {"meta-d'":1.472, "AUC2":0.742, "Criterion":12.185, "Confidence":0.482}
for m, name in enumerate(MEASURE_NAMES):
    data = lo_rb[:, :, m]
    complete = data[~np.any(np.isnan(data), axis=1)]
    if complete.shape[0] < 2:
        print(f"  {name:<20}      NaN      nan       {'':4} {'   NaN':>7}")
        continue
    F, df_b, df_e, p, eta2 = rm_anova_1way(complete)
    rep_f = REPORTED_T9.get(name, np.nan)
    match = ("✓" if abs(F-rep_f)<abs(rep_f)*0.05+0.05 else "~") if not np.isnan(rep_f) else ""
    print(f"  {name:<20} {F:9.3f} {p:8.4f} {p_stars(p):>4} {eta2:7.3f}  {match}")



Supplementary Table 9 — Response bias: Locke (n=10, 7 conditions)
  Measure                F(6,54)        p  sig     η²p
  -------------------------------------------------------
  meta-d'                  1.472   0.2053   ns   0.141  ✓
  AUC2                     0.742   0.6179   ns   0.076  ✓
  Gamma                    0.863   0.5281   ns   0.087  
  Phi                      0.927   0.4831   ns   0.093  
  DeltaConf                0.742   0.6179   ns   0.076  
  M-Ratio                  1.090   0.3800   ns   0.108  
  AUC2-Ratio               1.176   0.3329   ns   0.116  
  Gamma-Ratio              1.063   0.3962   ns   0.106  
  Phi-Ratio                1.078   0.3870   ns   0.107  
  DeltaConf-Ratio          1.072   0.3906   ns   0.106  
  M-Diff                   0.911   0.4939   ns   0.092  
  AUC2-Diff                1.119   0.3635   ns   0.111  
  Gamma-Diff               0.814   0.5641   ns   0.083  
  Phi-Diff                 1.149   0.3471   ns   0.113  
  DeltaConf-Diff    